# DDR Chart Generator — Training Notebook

**Before running: Runtime → Change runtime type → T4 GPU**

### Assumed setup
Your Google Drive already has StepMania packs uploaded at:
```
MyDrive/142B Group/ddc_project/data/raw/
  Some Pack Name/
    Song Name/
      Song.sm
      Song.mp3
  Another Pack/
    ...
```
Folder names don't matter. The code searches recursively for any folder
containing both a .sm file and an audio file (.mp3 / .ogg / .wav).

### Run order
**Every session:** re-run cells 1, 2, 3 (Colab VMs are wiped on disconnect).
**First time only:** run cells 4 onwards. Results save to Drive automatically.
**Resuming after disconnect:** re-run cells 1-3, then jump straight to cell 5 with `--curriculum_start` set to the last completed stage.

In [ ]:
# ── 1. Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p '/content/drive/MyDrive/142B Group/ddc_project/data/cache'
!mkdir -p '/content/drive/MyDrive/142B Group/ddc_project/data/raw'
!mkdir -p '/content/drive/MyDrive/142B Group/ddc_project/checkpoints'
!mkdir -p '/content/drive/MyDrive/142B Group/ddc_project/logs'
print('Drive mounted.')

In [ ]:
# ── 2. Clone repo and symlink data ──────────────────────────────────────
%cd /content
!git clone --branch helena https://github.com/alexseungum/ieor142b_project.git ddc-chart-gen
%cd /content/ddc-chart-gen

# Symlink Drive data folder so code finds it at 'data/'
!ln -sf '/content/drive/MyDrive/142B Group/ddc_project/data' data
print('Repo cloned and data symlinked.')

In [ ]:
!git pull origin helena
!git log --oneline -1 && echo "---" && git log --oneline -1 origin/helena

In [ ]:
# ── 3. Install dependencies ─────────────────────────────────────────────
!pip install librosa soundfile torchaudio --quiet
print('Done.')

In [ ]:
# ── 4. Verify data ────────────────────────────────────────────────────────────
import sys
from pathlib import Path
sys.path.insert(0, '/content/ddc-chart-gen')
from utils.data_utils import scan_song_dirs, parse_chart_file, build_sample

DATA_ROOT = '/content/drive/MyDrive/142B Group/ddc_project/data/raw'
usable_pairs, no_audio, pack_stats = scan_song_dirs(DATA_ROOT)

diff_order = ['beginner', 'easy', 'medium', 'hard', 'challenge']
print(f"{'='*60}\nDATASET SUMMARY\n{'='*60}")
print(f"Usable song-audio pairs : {len(usable_pairs)}")
print(f"Missing audio (skipped) : {len(no_audio)}")
if no_audio:
    for d in no_audio[:5]:
        print(f"  - {Path(d).relative_to(DATA_ROOT)}")
    if len(no_audio) > 5:
        print(f"  ... and {len(no_audio)-5} more")

print(f"\n{'='*60}\nBREAKDOWN BY PACK\n{'='*60}")
for pack, stats in sorted(pack_stats.items()):
    fmt_str = [f"{stats[k]} .{k}" for k in ('sm', 'ssc') if stats.get(k, 0)]
    no_aud  = f"  ⚠ {stats['no_audio']} missing audio" if stats['no_audio'] else ""
    print(f"\n  [{pack}]  {stats['songs']} songs ({', '.join(fmt_str)}){no_aud}")
    diff_counts = [f"{d[:3].upper()}:{stats['difficulties'].get(d, 0)}" for d in diff_order]
    print(f"    difficulties: {' | '.join(diff_counts)}")

if usable_pairs:
    audio_path, chart_path, fmt, pack = usable_pairs[0]
    sm_data = parse_chart_file(chart_path)
    sample  = build_sample(audio_path, chart_path)
    print(f"\n{'='*60}\nSANITY CHECK: {sm_data['title']}  [{fmt.upper()}]\n{'='*60}")
    if sample:
        T = sample['beat_frames'].shape[0]
        print(f"Timesteps : {T}  |  mel: {sample['mel'].shape}  |  "
              f"step density: {(sample['y'].sum(-1) > 0).mean():.1%}")
    else:
        print("build_sample returned None — no dance-single chart found")
else:
    print("No usable pairs found — check your data folder structure")

In [ ]:
# ── 4b. Build cache ───────────────────────────────────────────────────────────
# Builds base_train.pkl and base_val.pkl before training. Safe to re-run — skips if cache exists.
import sys
sys.path.insert(0, '/content/ddc-chart-gen')
from dataset import build_cache

DATA_ROOT = '/content/drive/MyDrive/142B Group/ddc_project/data/raw'
CACHE_DIR = '/content/drive/MyDrive/142B Group/ddc_project/data/cache'

build_cache(DATA_ROOT, CACHE_DIR)
print("Ready to train.")

In [ ]:
# ── 5. Train ─────────────────────────────────────────────────────────────
import os
DATA_ROOT = '/content/drive/MyDrive/142B Group/ddc_project/data/raw'
CACHE_DIR = '/content/drive/MyDrive/142B Group/ddc_project/data/cache'
CKPT_DIR  = '/content/drive/MyDrive/142B Group/ddc_project/checkpoints'
os.environ['DATA_ROOT'] = DATA_ROOT
os.environ['CACHE_DIR'] = CACHE_DIR
os.environ['CKPT_DIR']  = CKPT_DIR

# To resume from a checkpoint, set RESUME_CKPT to the checkpoint path.
# Set CURRICULUM_START to the stage you want to resume from (0-4).
# Leave RESUME_CKPT empty to train from scratch.
RESUME_CKPT      = f'{CKPT_DIR}/best_model.pt'   # set to '' to train from scratch
CURRICULUM_START = 4                               # stage to resume from

%cd /content/ddc-chart-gen
resume_flag = f'--resume "{RESUME_CKPT}"' if RESUME_CKPT else ''
!python train.py \
    --data_root      "$DATA_ROOT" \
    --cache_dir      "$CACHE_DIR" \
    --checkpoint_dir "$CKPT_DIR" \
    --curriculum_start {CURRICULUM_START} \
    --num_workers 0 \
    {resume_flag}


In [ ]:
# ── 5b. Check checkpoint before resuming ─────────────────────────────────
import torch
CKPT_DIR = '/content/drive/MyDrive/142B Group/ddc_project/checkpoints'
ckpt = torch.load(f'{CKPT_DIR}/best_model.pt', map_location='cpu')
stage = ckpt['stage']
print(f"Checkpoint: stage {stage}, epoch {ckpt['epoch']}, val_F1 {ckpt['val_f1']:.4f}, val_arrow_exact {ckpt.get('val_arrow_exact', float('nan')):.4f}")
print()
print(f"To resume: in cell 5, set RESUME_CKPT = f'{{CKPT_DIR}}/best_model.pt'  and  CURRICULUM_START = {stage}")
print("Then run cell 5.")


In [ ]:
# ── 6. Plot loss curves ───────────────────────────────────────────────────────
%cd /content/ddc-chart-gen
import json, sys
sys.path.insert(0, '/content/ddc-chart-gen')
from utils.plot_utils import plot_training_curves

with open('logs/train_history.json') as f:
    history = json.load(f)
plot_training_curves(history, save_path='logs/training_curves.png')
!mkdir -p '/content/drive/MyDrive/142B Group/ddc_project/logs'
!cp logs/training_curves.png '/content/drive/MyDrive/142B Group/ddc_project/logs/'
print('Saved to Drive.')

In [ ]:
# ── 6b. Seeded validation generation ─────────────────────────────────────────
# Loads a val song, seeds decoder with first N_SEED GT steps, then generates.
%cd /content/ddc-chart-gen
import os, sys, random, pickle, base64
import numpy as np
sys.path.insert(0, '/content/ddc-chart-gen')
from pathlib import Path
from google.colab import files
os.makedirs('logs', exist_ok=True)

from models.model import load_model
from generate import generate_seeded, build_chart_data
from utils.data_utils import find_audio_for_title, compute_step_metrics, compute_arrow_metrics
from utils.plot_utils import plot_generation
from visualizer import build_html

# ── Config ────────────────────────────────────────────────────────────────────
CKPT_DIR      = '/content/drive/MyDrive/142B Group/ddc_project/checkpoints'
CACHE_DIR     = '/content/drive/MyDrive/142B Group/ddc_project/data/cache'
DATA_ROOT     = '/content/drive/MyDrive/142B Group/ddc_project/data/raw'
TARGET_DIFF   = 2       # 0=Beginner  1=Easy  2=Medium  3=Hard  4=Challenge
TEMPERATURE   = 1.2
THRESHOLD     = 0.5
SUBDIV_SCALES = [1.0, 0.60, 0.50, 0.45]
N_SEED        = 16      # GT steps to seed before free-running (0 to disable)

SONG_TITLE    = ''      # <-- e.g. 'Chaos'  (leave '' for random)
PACK_NAME     = ''      # <-- e.g. 'DDR A'  (leave '' to search all packs)
DIFF_NAMES    = {0: 'Beginner', 1: 'Easy', 2: 'Medium', 3: 'Hard', 4: 'Challenge'}

# ── Load model ────────────────────────────────────────────────────────────────
model, ckpt = load_model(f'{CKPT_DIR}/best_model.pt')
device = next(model.parameters()).device
print(f"Loaded: stage {ckpt.get('stage', '?')}, epoch {ckpt.get('epoch', '?')}")

# ── Pick song ─────────────────────────────────────────────────────────────────
with open(f'{CACHE_DIR}/base_val.pkl', 'rb') as f:
    all_val = pickle.load(f)

_norm = lambda s: s.lower().strip()
if SONG_TITLE:
    candidates = [s for s in all_val
                  if _norm(s.get('title', '')) == _norm(SONG_TITLE)
                  and s['difficulty'] == TARGET_DIFF]
    if not candidates:
        candidates = [s for s in all_val if _norm(s.get('title', '')) == _norm(SONG_TITLE)]
    if not candidates:
        print(f"'{SONG_TITLE}' not found in val cache. Available titles:")
        for t in sorted({s.get('title', '') for s in all_val}):
            print(f"  '{t}'")
        raise ValueError(f"'{SONG_TITLE}' not in val cache")
    song = candidates[0]
    if len(candidates) > 1:
        print(f"  {len(candidates)} matches — using diff={song['difficulty']}")
else:
    candidates = [s for s in all_val if s['difficulty'] == TARGET_DIFF] or all_val
    song = random.choice(candidates)

title = song.get('title', 'unknown')
print(f"Song: '{title}'  diff={song['difficulty']}  timesteps={song['y'].shape[0]}")

# ── Generate ──────────────────────────────────────────────────────────────────
step_mask, arrow_preds, step_probs, seed_cutoff = generate_seeded(
    model, song, device,
    temperature=TEMPERATURE, threshold=THRESHOLD,
    subdiv_scales=SUBDIV_SCALES, n_seed=N_SEED,
)
T    = len(step_mask)
y_np = song['y'][:T].astype(np.float32)
print(f"Generated: {step_mask.sum()} steps / {T} timesteps  ({100*step_mask.mean():.1f}% density)")

# ── Step metrics (generated portion only) ────────────────────────────────────
m = compute_step_metrics(step_mask, y_np, start=seed_cutoff)
print(f"\nStep placement [{seed_cutoff}:{T}]")
print(f"  F1={m['f1']:.3f}  P={m['precision']:.3f}  R={m['recall']:.3f}  "
      f"pred={m['tp']+m['fp']}  gt={m['tp']+m['fn']}")

# ── Arrow metrics (at TP positions) ──────────────────────────────────────────
am = compute_arrow_metrics(step_mask, arrow_preds, y_np)
print(f"\nArrow quality (at {am['n_tp']} TP steps)  [seeded AR]")
print(f"  Exact match  : {am['arrow_exact']*100:.1f}%")
print(f"  Per-direction: L={am['per_dir_acc'][0]*100:.1f}%  D={am['per_dir_acc'][1]*100:.1f}%  "
      f"U={am['per_dir_acc'][2]*100:.1f}%  R={am['per_dir_acc'][3]*100:.1f}%")
pd, gd = am['pred_combo_dist'], am['gt_combo_dist']
print(f"  Combo dist   :        pred    GT")
for k in ('single', 'bracket', 'triple', 'quad'):
    print(f"    {k:<8}: {pd.get(k,0)*100:5.1f}%  {gd.get(k,0)*100:5.1f}%")

# ── Plot ──────────────────────────────────────────────────────────────────────
subdiv_thresh = [THRESHOLD * s for s in SUBDIV_SCALES]
plot_generation(step_probs, y_np, arrow_preds, song['subdiv_types'][:T],
                title=f"{title} (seeded AR)", subdiv_thresh=subdiv_thresh,
                seed_cutoff=seed_cutoff, save_path='logs/seeded_generation.png')

# ── Build & download visualizer ───────────────────────────────────────────────
audio_path, song_bpm, song_offset = find_audio_for_title(DATA_ROOT, title, pack_name=PACK_NAME)
chart_data = build_chart_data(step_mask, arrow_preds, bpm=song_bpm, title=title,
                               difficulty=song['difficulty'], offset=-song_offset)
audio_data_uri = None
if audio_path:
    mime_map = {'.mp3': 'audio/mpeg', '.ogg': 'audio/ogg', '.wav': 'audio/wav'}
    suffix   = Path(audio_path).suffix.lower()
    audio_data_uri = (f"data:{mime_map[suffix]};base64,"
                      + base64.b64encode(open(audio_path, 'rb').read()).decode())
    print(f"\nAudio: {Path(audio_path).name}  BPM={song_bpm:.2f}  offset={song_offset:.4f}s")
else:
    print("\nAudio not found — visualizer loads without playback")

html = build_html(chart_data, audio_data_uri=audio_data_uri)
with open('logs/seeded_visualizer.html', 'w') as f:
    f.write(html)
files.download('logs/seeded_visualizer.html')
print(f"Downloaded visualizer — {len(chart_data['events'])} steps")

In [ ]:
# ── 6c. No-AR ablation — full validation sweep ───────────────────────────────
%cd /content/ddc-chart-gen
import os, sys, pickle
import numpy as np
sys.path.insert(0, '/content/ddc-chart-gen')

from models.model import load_model
from generate import generate_seeded, generate_no_ar
from utils.data_utils import compute_arrow_metrics

CKPT_DIR      = '/content/drive/MyDrive/142B Group/ddc_project/checkpoints'
CACHE_DIR     = '/content/drive/MyDrive/142B Group/ddc_project/data/cache'
TARGET_DIFF   = 2       # set to None to include all difficulties
TEMPERATURE   = 1.2
THRESHOLD     = 0.5
SUBDIV_SCALES = [1.0, 0.60, 0.50, 0.45]
N_SEED        = 16

model, ckpt = load_model(f'{CKPT_DIR}/best_model.pt')
device = next(model.parameters()).device
print(f"Loaded: stage {ckpt.get('stage','?')}, epoch {ckpt.get('epoch','?')}")

with open(f'{CACHE_DIR}/base_val.pkl', 'rb') as f:
    all_val = pickle.load(f)

songs = [s for s in all_val if TARGET_DIFF is None or s['difficulty'] == TARGET_DIFF]
print(f"Running ablation on {len(songs)} songs (diff={TARGET_DIFF})\n")

rows = []
for song in songs:
    title = song.get('title', 'unknown')

    # AR baseline — T comes from the generator, not song['y']
    sm_ar, ap_ar, _, _ = generate_seeded(
        model, song, device,
        temperature=TEMPERATURE, threshold=THRESHOLD,
        subdiv_scales=SUBDIV_SCALES, n_seed=N_SEED,
    )
    y_ar = song['y'][:len(sm_ar)].astype(np.float32)
    am_ar = compute_arrow_metrics(sm_ar, ap_ar, y_ar)

    # No-AR ablation
    sm_noar, ap_noar, _ = generate_no_ar(
        model, song, device,
        temperature=TEMPERATURE, threshold=THRESHOLD, subdiv_scales=SUBDIV_SCALES,
    )
    y_noar = song['y'][:len(sm_noar)].astype(np.float32)
    am_noar = compute_arrow_metrics(sm_noar, ap_noar, y_noar)

    rows.append({
        'title': title,
        'diff': song['difficulty'],
        'ar_exact':    am_ar['arrow_exact'],
        'noar_exact':  am_noar['arrow_exact'],
        'ar_per_dir':   am_ar['per_dir_acc'],
        'noar_per_dir': am_noar['per_dir_acc'],
    })
    print(f"  {title:<30} AR={am_ar['arrow_exact']*100:4.1f}%  no-AR={am_noar['arrow_exact']*100:4.1f}%")

# ── Aggregate ─────────────────────────────────────────────────────────────────
ar_exact   = np.array([r['ar_exact']   for r in rows])
noar_exact = np.array([r['noar_exact'] for r in rows])
ar_pd      = np.stack([r['ar_per_dir']   for r in rows])   # (N, 4)
noar_pd    = np.stack([r['noar_per_dir'] for r in rows])   # (N, 4)
dirs = ['L', 'D', 'U', 'R']

print(f"\n{'='*55}")
print(f"{'Metric':<28} {'AR':>8}  {'no-AR':>8}")
print(f"{'-'*55}")
print(f"{'Exact match (mean)':<28} {ar_exact.mean()*100:7.1f}%  {noar_exact.mean()*100:7.1f}%")
print(f"{'Exact match (std)':<28} {ar_exact.std()*100:7.1f}%  {noar_exact.std()*100:7.1f}%")
for i, d in enumerate(dirs):
    print(f"{'Per-dir acc ' + d:<28} {ar_pd[:,i].mean()*100:7.1f}%  {noar_pd[:,i].mean()*100:7.1f}%")
print(f"{'='*55}")
print(f"Songs: {len(rows)}  |  diff filter: {TARGET_DIFF}")


In [ ]:
# ── 6c. Visualize a GT chart by song title ────────────────────────────────────
# Set SONG_TITLE (case-insensitive). Optionally set GT_DIFFICULTY to filter charts.
%cd /content/ddc-chart-gen
import base64, sys
from pathlib import Path
sys.path.insert(0, '/content/ddc-chart-gen')
from utils.data_utils import parse_chart_file, find_audio_for_title
from visualizer import build_chart_json, build_html
from google.colab import files

DATA_ROOT     = '/content/drive/MyDrive/142B Group/ddc_project/data/raw'
SONG_TITLE    = 'PUT SONG TITLE HERE'   # <-- edit this (case-insensitive)
GT_DIFFICULTY = None                    # e.g. 'medium', or None for first chart found

mime_map = {'.mp3': 'audio/mpeg', '.ogg': 'audio/ogg', '.wav': 'audio/wav'}

# ── Find chart ────────────────────────────────────────────────────────────────
def _norm(s): return s.lower().strip()
all_charts = list(Path(DATA_ROOT).rglob('*.sm')) + list(Path(DATA_ROOT).rglob('*.ssc'))
print(f"Scanning {len(all_charts)} chart files...")

match = None
for chart_path in all_charts:
    try:
        sm_data = parse_chart_file(str(chart_path))
        if _norm(sm_data['title']) == _norm(SONG_TITLE):
            match = (chart_path, sm_data)
            break
    except Exception:
        pass

if match is None:
    print(f"'{SONG_TITLE}' not found. Available titles:")
    seen = set()
    for chart_path in all_charts:
        try:
            t = parse_chart_file(str(chart_path))['title']
            if t not in seen:
                print(f"  '{t}'")
                seen.add(t)
        except Exception:
            pass
else:
    chart_path, _ = match
    chart_data = build_chart_json(str(chart_path), difficulty_filter=GT_DIFFICULTY)
    print(f"  Diff: {chart_data['difficulty']}  BPM: {chart_data['bpm']:.1f}  "
          f"Offset: {chart_data['offset']:.4f}s  Steps: {chart_data['total_steps']}")

    audio_path, _, _ = find_audio_for_title(DATA_ROOT, SONG_TITLE)
    audio_data_uri = None
    if audio_path:
        suffix = Path(audio_path).suffix.lower()
        audio_data_uri = (f"data:{mime_map[suffix]};base64,"
                          + base64.b64encode(open(audio_path, 'rb').read()).decode())
        print(f"  Audio: {Path(audio_path).name}")
    else:
        print("  Audio not found — visualizer loads without playback")

    html = build_html(chart_data, audio_data_uri=audio_data_uri)
    out  = 'logs/gt_visualizer.html'
    with open(out, 'w') as f:
        f.write(html)
    files.download(out)
    print(f"Downloaded {out}")

In [ ]:
# ── 7. Generate a chart ───────────────────────────────────────────────────────
%cd /content/ddc-chart-gen
import os, sys
import numpy as np
sys.path.insert(0, '/content/ddc-chart-gen')
from utils.plot_utils import plot_chart_preview
from config import VALID_SUBDIV_POSITIONS, N_VALID_PER_MEASURE, SUBDIVISION
from utils.data_utils import get_subdiv_type

# Option B: upload any song from your computer (mp3, ogg, wav)
from google.colab import files as colab_files
uploaded   = colab_files.upload()
test_audio = list(uploaded.keys())[0]
print(f'Using: {test_audio}')

# ── Set BPM manually — check the song's .sm/.ssc file or use a BPM finder ────
SONG_BPM      = 123.0   # <-- edit this
THRESHOLD     = 0.5
SUBDIV_SCALES = [1.0, 0.60, 0.50, 0.45]
_eff = [THRESHOLD * s for s in SUBDIV_SCALES]
print(f"Thresholds — 4th:{_eff[0]:.3f}  8th:{_eff[1]:.3f}  12th:{_eff[2]:.3f}  16th:{_eff[3]:.3f}")

CKPT_DIR = '/content/drive/MyDrive/142B Group/ddc_project/checkpoints'
os.environ['TEST_AUDIO']   = test_audio
os.environ['SONG_BPM']     = str(SONG_BPM)
os.environ['SUBDIV_SCALES'] = ' '.join(str(s) for s in SUBDIV_SCALES)

# Adjust --difficulty (0-4), --temperature (<1=greedier, >1=more diverse)
!python generate.py \
    --audio      "$TEST_AUDIO" \
    --checkpoint "$CKPT_DIR/best_model.pt" \
    --bpm        "$SONG_BPM" \
    --difficulty 2 \
    --temperature 1.2 \
    --threshold  {THRESHOLD} \
    --subdiv_scales $SUBDIV_SCALES \
    --output     output_chart

!cp -r output_chart '/content/drive/MyDrive/142B Group/ddc_project/'

# ── Plot ──────────────────────────────────────────────────────────────────────
preds       = np.load('output_chart/predictions.npz')
step_probs  = preds.get('step_probs', np.zeros(len(preds['step_mask'])))
T           = len(preds['step_mask'])
subdiv_arr  = np.array([get_subdiv_type(VALID_SUBDIV_POSITIONS[t % N_VALID_PER_MEASURE], SUBDIVISION)
                        for t in range(T)])
plot_chart_preview(step_probs, preds['arrow_preds'], subdiv_arr, _eff,
                   title=test_audio, save_path='output_chart/chart_preview.png')
print('Done! Run cell 9 to download visualizer.html')

In [ ]:
# ── 8. Threshold sweep ────────────────────────────────────────────────────────
%cd /content/ddc-chart-gen
import os, sys
import numpy as np
sys.path.insert(0, '/content/ddc-chart-gen')
from models.model import load_model, generate_chart
from generate import audio_to_model_input
from utils.plot_utils import plot_threshold_sweep

CKPT_DIR = '/content/drive/MyDrive/142B Group/ddc_project/checkpoints'
model, _ = load_model(f'{CKPT_DIR}/best_model.pt')
device   = str(next(model.parameters()).device)

X, subdiv_types = audio_to_model_input(test_audio, bpm=float(os.environ['SONG_BPM']))

thresholds = np.linspace(0.1, 0.9, 17)
densities  = []
for th in thresholds:
    mask, _, _ = generate_chart(model, X, subdiv_types, difficulty=2,
                                threshold=float(th), device=device)
    densities.append(mask.mean())
    print(f"  th={th:.2f}  density={mask.mean()*100:.1f}%")

plot_threshold_sweep(thresholds, densities, save_path='logs/threshold_sweep.png')
!mkdir -p '/content/drive/MyDrive/142B Group/ddc_project/logs'
!cp logs/threshold_sweep.png '/content/drive/MyDrive/142B Group/ddc_project/logs/'

In [ ]:
# ── 8b. Sanity check: step & arrow probability distributions ──────────────────
%cd /content/ddc-chart-gen
import os, sys, torch
import numpy as np
sys.path.insert(0, '/content/ddc-chart-gen')
from models.model import load_model
from generate import audio_to_model_input
from utils.plot_utils import plot_prob_sanity

CKPT_DIR = '/content/drive/MyDrive/142B Group/ddc_project/checkpoints'
model, _ = load_model(f'{CKPT_DIR}/best_model.pt')
device   = next(model.parameters()).device

X, subdiv_types = audio_to_model_input(test_audio, bpm=float(os.environ['SONG_BPM']))
X, subdiv_types = X.to(device), subdiv_types.to(device)
diff_tensor = torch.tensor([2], dtype=torch.long, device=device)

with torch.no_grad():
    encoder_out  = model.encode(X, diff_tensor, subdiv_types)
    step_logits  = model.step_head(encoder_out)
    arrows_zeros = torch.zeros(1, encoder_out.shape[1], 4, device=device)
    arrow_logits = model.decoder(arrows_zeros, encoder_out)

step_probs  = torch.sigmoid(step_logits.squeeze(-1).squeeze(0)).cpu().numpy()
arrow_probs = torch.sigmoid(arrow_logits.squeeze(0)).cpu().numpy()
plot_prob_sanity(step_probs, arrow_probs, save_path='logs/prob_sanity.png')

In [ ]:
# ── 9. Download results ──────────────────────────────────────────────────
%cd /content/ddc-chart-gen
CKPT_DIR = '/content/drive/MyDrive/142B Group/ddc_project/checkpoints'

from google.colab import files
files.download('output_chart/visualizer.html')
files.download('output_chart/chart.sm')
files.download(f'{CKPT_DIR}/best_model.pt')
files.download('logs/training_curves.png')
files.download('logs/threshold_sweep.png')